# Stage 2 – Exploratory Data Analysis & Data Quality Assessment
**Kenyan customer-support sentiment dataset** (synthetic, 30,000 rows × 40 columns).

Principles: **no records are deleted or modified in place.** Every proposed cleaning action is logged and explained, and left for a human decision.

## 1. Environment setup & first look

In [ ]:
import warnings, re, os, glob, json
warnings.filterwarnings("ignore")
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.neighbors import NearestNeighbors

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)

CANDIDATE_PATHS = [
    "/kaggle/input/datasets/linzkibet/kenya-sentiment-analysis-dataset/kenyan_customer_support_sentiment_dataset.csv",
    "/kaggle/input/kenya-sentiment-analysis-dataset/kenyan_customer_support_sentiment_dataset.csv",
    "/kaggle/working/kenyan_customer_support_sentiment_dataset.csv",
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if DATA_PATH is None:
    hits = glob.glob("/kaggle/input/**/kenyan_customer_support_sentiment_dataset.csv", recursive=True)
    DATA_PATH = hits[0] if hits else None
assert DATA_PATH, "Dataset not found. Add it via 'Add Data' in the Kaggle sidebar."
print("Loading:", DATA_PATH)
df = pd.read_csv(DATA_PATH)

print("Dimensions:", df.shape)
display(df.head(10))
display(df.tail(5))
print("\nColumns:", list(df.columns))
display(df.dtypes.to_frame("dtype"))
display(df.describe(include="all").T)

### Column mapping
Columns are located by name so the notebook tolerates small naming differences. Check the printout for anything marked `None`.

In [ ]:
def find_col(*names):
    lower = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in lower:
            return lower[n.lower()]
    return None

SENT    = find_col("sentiment", "sentiment_label")
EMO     = find_col("emotion")
PRI     = find_col("priority")
DEPT    = find_col("department")
INQ     = find_col("inquiry_type")
LANG    = find_col("language", "language_category")
LANGVAR = find_col("language_variant", "language_type", "dialect")
STATUS  = find_col("resolution_status")
QUEUE   = find_col("queue")
CHANNEL = find_col("customer_channel", "channel")
TEXT    = find_col("customer_inquiry")
CTX     = find_col("conversation_context")
TID     = find_col("ticket_id")
SPLIT   = find_col("split", "data_split", "dataset_split", "subset")
SCORE   = find_col("sentiment_score")
INTENS  = find_col("emotional_intensity")
RESTIME = find_col("resolution_time_minutes")
CONF    = find_col("annotation_confidence")
COMPLEX = find_col("linguistic_complexity")
FLAG_NAMES = ["contains_sarcasm","contains_negation","contains_slang","contains_ambiguity",
              "contains_context_dependency","contains_typos","contains_emoji"]
FLAGS = [c for c in FLAG_NAMES if c in df.columns]

mapping = dict(SENT=SENT, EMO=EMO, PRI=PRI, DEPT=DEPT, INQ=INQ, LANG=LANG, LANGVAR=LANGVAR, STATUS=STATUS,
               QUEUE=QUEUE, CHANNEL=CHANNEL, TEXT=TEXT, CTX=CTX, TID=TID, SPLIT=SPLIT, SCORE=SCORE,
               INTENS=INTENS, RESTIME=RESTIME, CONF=CONF, COMPLEX=COMPLEX)
for k, v in mapping.items():
    print(f"{k:8s} -> {v}")
print("Flag columns found:", FLAGS)
print("Flag columns missing:", [c for c in FLAG_NAMES if c not in df.columns])

issues = []   # running log of data-quality findings and proposed actions

# Analysis-only helper columns (originals are untouched)
df["_sent"] = df[SENT].astype(str).str.strip().str.title()
ORDER = [c for c in ["Positive","Negative","Neutral","Mixed"] if c in set(df["_sent"])]
ORDER += [c for c in df["_sent"].unique() if c not in ORDER]

def to_bool(s):
    m = {"true":True,"false":False,"1":True,"0":False,"yes":True,"no":False,"1.0":True,"0.0":False}
    return s.astype(str).str.strip().str.lower().map(m)

def heat(ct, title, fmt=".1f", cmap="YlGnBu", figsize=(9, 5)):
    plt.figure(figsize=figsize)
    sns.heatmap(ct, annot=True, fmt=fmt, cmap=cmap, linewidths=.5)
    plt.title(title); plt.tight_layout(); plt.show()

def pct_ct(a, b):
    """Row-normalised crosstab (% of each row in each column of b)."""
    return pd.crosstab(df[a], df[b], normalize="index") * 100

def cramers_v(a, b):
    ct = pd.crosstab(df[a], df[b])
    chi2 = chi2_contingency(ct)[0]
    n = ct.values.sum()
    return np.sqrt((chi2 / n) / (min(ct.shape) - 1))

## 2. Data quality assessment
### 2A. Missing values

In [ ]:
blank = {c: int((df[c].astype(str).str.strip() == "").sum()) for c in df.select_dtypes("object").columns}
miss = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "blank_strings": pd.Series(blank).reindex(df.columns).fillna(0).astype(int),
}).sort_values("missing", ascending=False)
display(miss)

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
miss_nz = miss[miss["missing"] > 0]
if len(miss_nz):
    miss_nz["missing_%"].sort_values().plot.barh(ax=ax[0], color="indianred")
    ax[0].set_title("Missing values (%) – columns with gaps"); ax[0].set_xlabel("%")
else:
    ax[0].text(.5, .5, "No missing values", ha="center", va="center"); ax[0].set_axis_off()
sns.heatmap(df.sample(min(2000, len(df)), random_state=1).isna(), cbar=False, yticklabels=False, ax=ax[1])
ax[1].set_title("Missingness map (2,000-row sample; dark = missing)")
plt.tight_layout(); plt.show()

attention = miss[(miss["missing"] > 0) | (miss["blank_strings"] > 0)]
print("Columns requiring attention:", list(attention.index) or "none")
for c, r in attention.iterrows():
    issues.append(f"[missing] {c}: {r['missing']} NaN ({r['missing_%']}%), {r['blank_strings']} blank strings. "
                  f"Proposal: confirm whether the gap is structural (e.g. no conversation_context for first message) before imputing.")

### 2B. Duplicates & near-duplicates

In [ ]:
print("Fully duplicated rows:", df.duplicated().sum())
if TID:
    print(f"Duplicate {TID}:", df[TID].duplicated().sum(), "| unique:", df[TID].nunique(), "/", len(df))
    if df[TID].duplicated().sum():
        issues.append(f"[duplicates] {df[TID].duplicated().sum()} duplicate {TID} values. Proposal: inspect; keep one only if rows are identical.")

norm = lambda s: re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", str(s).lower())).strip()
df["_norm_text"] = df[TEXT].map(norm)
print("Duplicate customer inquiries (exact):", df[TEXT].duplicated().sum())
print("Duplicate after normalising case/punctuation:", df["_norm_text"].duplicated().sum())
print(f"Unique normalised texts: {df['_norm_text'].nunique()} / {len(df)} "
      f"({df['_norm_text'].nunique()/len(df):.1%}). Low values suggest heavy templating.")

# Same text, different sentiment label = label conflict (important for training!)
conf_ = df.groupby("_norm_text")["_sent"].nunique()
conflicts = conf_[conf_ > 1]
print("Identical texts carrying different sentiment labels:", len(conflicts))
if len(conflicts):
    display(df[df["_norm_text"].isin(conflicts.index[:5])].sort_values("_norm_text")[[TEXT, SENT] + ([SCORE] if SCORE else [])].head(15))
    issues.append(f"[label conflict] {len(conflicts)} distinct texts appear with >1 sentiment label. Proposal: review manually; may be legitimate context-dependent cases.")

# Near-duplicates: TF-IDF cosine similarity on a sample of unique texts
uniq = df.drop_duplicates("_norm_text")
samp = uniq.sample(min(6000, len(uniq)), random_state=42)
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1).fit_transform(samp["_norm_text"])
nn = NearestNeighbors(n_neighbors=2, metric="cosine", algorithm="brute").fit(vec)
dist, ind = nn.kneighbors(vec)
sim = 1 - dist[:, 1]
near = np.where(sim >= 0.90)[0]
print(f"Near-duplicate candidates (cosine ≥ 0.90) in a {len(samp)}-text sample: {len(near)} ({len(near)/len(samp):.1%})")
pairs = pd.DataFrame({"text_a": samp[TEXT].values[near], "text_b": samp[TEXT].values[ind[near, 1]],
                      "similarity": sim[near].round(3)}).head(10)
display(pairs)
if len(near)/len(samp) > .05:
    issues.append(f"[near-duplicates] ~{len(near)/len(samp):.0%} of sampled texts have a ≥0.90 cosine neighbour. "
                  "Proposal: group near-duplicates and ensure they do NOT straddle train/val/test (leakage).")

# Leakage check across splits
if SPLIT:
    sp = df.groupby("_norm_text")[SPLIT].nunique()
    print("Texts appearing in more than one split (leakage):", int((sp > 1).sum()))
    if (sp > 1).sum():
        issues.append(f"[leakage] {(sp>1).sum()} identical texts appear in multiple splits. Proposal: re-split by text group.")

### 2C. Data types

In [ ]:
bool_like = [c for c in df.columns if c.startswith(("contains_", "is_", "has_"))]
ok = {"true","false","0","1","yes","no","0.0","1.0"}
rows = []
for c in bool_like:
    vals = set(df[c].dropna().astype(str).str.strip().str.lower().unique())
    rows.append({"column": c, "dtype": str(df[c].dtype), "values": sorted(vals)[:6], "valid_boolean": vals <= ok})
display(pd.DataFrame(rows))
for r in rows:
    if not r["valid_boolean"]:
        issues.append(f"[dtype] {r['column']} is not a clean boolean {r['values']}. Proposal: map to 0/1 explicitly.")

num_expected = [c for c in [SCORE, INTENS, RESTIME, CONF] if c]
rows = []
for c in num_expected:
    coerced = pd.to_numeric(df[c], errors="coerce")
    rows.append({"column": c, "dtype": str(df[c].dtype), "non_numeric_values": int(coerced.isna().sum() - df[c].isna().sum()),
                 "min": coerced.min(), "max": coerced.max()})
display(pd.DataFrame(rows))

# Object columns that look numeric / possible mistyped columns
for c in df.select_dtypes("object").columns:
    if c.startswith("_"): continue
    conv = pd.to_numeric(df[c], errors="coerce")
    if conv.notna().mean() > 0.95:
        print(f"⚠ '{c}' is object dtype but ≥95% numeric")
        issues.append(f"[dtype] {c} stored as text but numeric. Proposal: cast to numeric.")

# Timestamps
ts_cols = [c for c in df.columns if re.search(r"timestamp|date|created|_at$|_time$", c.lower()) and "minute" not in c.lower()]
print("Timestamp-like columns:", ts_cols or "none")
for c in ts_cols:
    parsed = pd.to_datetime(df[c], errors="coerce")
    print(f"{c}: unparsable={parsed.isna().sum() - df[c].isna().sum()}, range={parsed.min()} → {parsed.max()}, "
          f"future-dated={(parsed > pd.Timestamp.now()).sum()}")

### 2D. Categorical consistency

In [ ]:
cat_cols = [c for c in [SENT, EMO, PRI, DEPT, INQ, LANG, LANGVAR, STATUS, QUEUE, CHANNEL] if c]
for c in cat_cols:
    vc = df[c].value_counts(dropna=False)
    print(f"\n=== {c}  ({df[c].nunique()} unique) ===")
    print(vc.head(25).to_string())
    groups = df[c].dropna().astype(str).groupby(lambda i: None)  # placeholder to keep API simple
    raw = pd.Series(df[c].dropna().astype(str).unique())
    key = raw.str.lower().str.replace(r"[\s_\-]+", "", regex=True)
    dups = raw.groupby(key).apply(list)
    dups = dups[dups.map(len) > 1]
    if len(dups):
        print("⚠ Inconsistent spelling/capitalisation:", dups.tolist())
        issues.append(f"[categorical] {c}: variants of the same value {dups.tolist()}. Proposal: standardise to one canonical form.")
    if (df[c].dropna().astype(str) != df[c].dropna().astype(str).str.strip()).any():
        issues.append(f"[categorical] {c}: leading/trailing whitespace. Proposal: strip.")

### Data-quality summary & proposed cleaning actions
Nothing below has been applied. Each item needs a decision.

In [ ]:
print(f"{len(issues)} finding(s):\n")
for i, s in enumerate(issues, 1):
    print(f"{i:2d}. {s}")
if not issues:
    print("No structural issues detected by these checks.")

## 3. Sentiment distribution

In [ ]:
vc = df["_sent"].value_counts().reindex(ORDER)
dist_tbl = pd.DataFrame({"count": vc, "percent": (vc / vc.sum() * 100).round(2)})
display(dist_tbl)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(x=vc.index, y=vc.values, ax=ax[0])
for i, v in enumerate(vc.values):
    ax[0].text(i, v, f"{v:,}", ha="center", va="bottom")
ax[0].set_title("Sentiment class counts"); ax[0].set_ylabel("records")
ax[1].pie(vc.values, labels=vc.index, autopct="%1.1f%%", startangle=90)
ax[1].set_title("Sentiment class share")
plt.tight_layout(); plt.show()

ratio = vc.max() / vc.min()
print(f"Majority/minority ratio: {ratio:.2f}  (majority={vc.idxmax()}, minority={vc.idxmin()})")
print("Assessment:", "balanced (<1.5)" if ratio < 1.5 else "mild imbalance (1.5–3): use stratified splits + macro-F1" if ratio < 3
      else "strong imbalance (>3): consider class weights / resampling")
print("Macro-F1 baseline for a majority-class predictor would have accuracy =", f"{vc.max()/vc.sum():.1%}")

In [ ]:
if SPLIT:
    print("Split sizes:"); print(df[SPLIT].value_counts().to_string())
    ct = pd.crosstab(df[SPLIT], df["_sent"])[ORDER]
    display(ct)
    heat(ct.div(ct.sum(1), axis=0) * 100, "Sentiment % within each split", figsize=(8, 3.5))
    chi2, p, *_ = chi2_contingency(ct)
    print(f"Chi-square test of split × sentiment independence: p = {p:.4f} "
          f"({'class mix differs across splits – investigate' if p < 0.05 else 'class mix consistent across splits'})")
else:
    print("⚠ No split column detected (looked for split / data_split / dataset_split / subset). "
          "Expected 21,000 / 4,500 / 4,500. Add or locate the split column to verify stratification.")

## 4. Language analysis

In [ ]:
lang_cols = [c for c in [LANG, LANGVAR] if c]
for c in lang_cols:
    print(f"\n{c}:"); print(df[c].value_counts().to_string())

for c in lang_cols:
    fig, ax = plt.subplots(1, 2, figsize=(16, 5))
    order = df[c].value_counts().index
    sns.countplot(y=df[c], order=order, ax=ax[0]); ax[0].set_title(f"Distribution of {c}")
    ct = pd.crosstab(df[c], df["_sent"], normalize="index")[ORDER] * 100
    ct.loc[order].plot.barh(stacked=True, ax=ax[1], width=.85); ax[1].set_title(f"Sentiment mix within each {c} (%)")
    ax[1].legend(loc="center left", bbox_to_anchor=(1, .5)); ax[1].invert_yaxis()
    plt.tight_layout(); plt.show()
    heat(pd.crosstab(df[c], df["_sent"])[ORDER], f"{c} × sentiment (counts)", fmt="d", figsize=(8, max(3, .45*df[c].nunique())))

In [ ]:
MIN_CELL = 100
for c in lang_cols:
    ct = pd.crosstab(df[c], df["_sent"])[ORDER]
    weak = [(i, j, int(ct.loc[i, j])) for i in ct.index for j in ct.columns if ct.loc[i, j] < MIN_CELL]
    print(f"{c}: cells with < {MIN_CELL} examples -> {weak or 'none'}")
    share = ct.div(ct.sum(1), axis=0)
    skew = share[(share.max(1) > 0.6)]
    if len(skew):
        print(f"   ⚠ categories where one sentiment exceeds 60% (language may leak the label):\n{(skew*100).round(1).to_string()}")
    print(f"   Cramér's V (language ↔ sentiment): {cramers_v(c, '_sent'):.3f}  (≈0 = independent, >0.3 = strong association)")

## 5. Linguistic complexity

In [ ]:
for c in FLAGS:
    df[f"_b_{c}"] = to_bool(df[c])
freq = pd.DataFrame({
    "count": {c: int(df[f"_b_{c}"].sum()) for c in FLAGS},
    "percent": {c: round(df[f"_b_{c}"].mean() * 100, 2) for c in FLAGS},
})
display(freq)
plt.figure(figsize=(8, 4)); freq["percent"].sort_values().plot.barh(color="teal")
plt.xlabel("% of records"); plt.title("Linguistic feature prevalence"); plt.tight_layout(); plt.show()
for c in FLAGS:
    n = freq.loc[c, "count"]
    if freq.loc[c, "percent"] < 5 or n < 500:
        print(f"⚠ {c}: only {n} examples ({freq.loc[c,'percent']}%) – may be too rare for the model to learn.")

In [ ]:
if FLAGS:
    by_sent = pd.DataFrame({c: df.groupby("_sent")[f"_b_{c}"].mean() * 100 for c in FLAGS}).loc[ORDER]
    heat(by_sent, "% of each sentiment class containing each feature", figsize=(11, 4))
    for c in lang_cols:
        by_lang = pd.DataFrame({f: df.groupby(c)[f"_b_{f}"].mean() * 100 for f in FLAGS})
        heat(by_lang, f"% of each {c} containing each feature", figsize=(11, max(3, .5*len(by_lang))))

In [ ]:
if COMPLEX:
    s = df[COMPLEX]
    if pd.api.types.is_numeric_dtype(s):
        display(s.describe().to_frame().T)
        fig, ax = plt.subplots(1, 2, figsize=(13, 4))
        sns.histplot(s, kde=True, ax=ax[0]); sns.boxplot(data=df, x="_sent", y=COMPLEX, order=ORDER, ax=ax[1])
        ax[0].set_title("linguistic_complexity"); ax[1].set_title("by sentiment"); plt.tight_layout(); plt.show()
        for c in lang_cols:
            plt.figure(figsize=(10, 4)); sns.boxplot(data=df, y=c, x=COMPLEX); plt.title(f"complexity by {c}"); plt.tight_layout(); plt.show()
    else:
        print(s.value_counts(normalize=True).mul(100).round(2).to_string())
        heat(pct_ct("_sent", COMPLEX).loc[ORDER], "linguistic_complexity % within each sentiment", figsize=(8, 4))
        for c in lang_cols:
            heat(pct_ct(c, COMPLEX), f"linguistic_complexity % within each {c}", figsize=(8, max(3, .45*df[c].nunique())))

### Manual inspection: 10 examples per phenomenon
Read these. For each, ask: *does the label match what the customer actually means?*

In [ ]:
def contains(col, pat):
    return df[col].astype(str).str.contains(pat, case=False, na=False) if col else pd.Series(False, index=df.index)

masks = {
    "Sarcasm": df.get("_b_contains_sarcasm", pd.Series(False, index=df.index)).fillna(False).astype(bool),
    "Negation": df.get("_b_contains_negation", pd.Series(False, index=df.index)).fillna(False).astype(bool),
    "Sheng": contains(LANG, "sheng") | contains(LANGVAR, "sheng"),
    "Code-switching": contains(LANG, "code|mix") | contains(LANGVAR, "code|mix"),
    "Ambiguity": df.get("_b_contains_ambiguity", pd.Series(False, index=df.index)).fillna(False).astype(bool),
    "Context-dependent": df.get("_b_contains_context_dependency", pd.Series(False, index=df.index)).fillna(False).astype(bool),
    "Mixed sentiment": df["_sent"] == "Mixed",
}
show_cols = [c for c in [TEXT, CTX, SENT, SCORE, EMO, LANG, LANGVAR] if c]

def show_examples(name, n=10):
    m = masks[name]
    print(f"\n{'='*100}\n{name.upper()}  –  {int(m.sum())} matching records\n{'='*100}")
    if m.sum() == 0:
        print("No matches (check column values)."); return
    for i, (_, r) in enumerate(df[m].sample(min(n, int(m.sum())), random_state=7).iterrows(), 1):
        meta = " | ".join(f"{c}={r[c]}" for c in show_cols[2:])
        print(f"{i:2d}. {r[TEXT]}")
        if CTX and pd.notna(r[CTX]) and str(r[CTX]).strip():
            print(f"    context: {r[CTX]}")
        print(f"    → {meta}")

show_examples("Sarcasm"); show_examples("Negation")

In [ ]:
show_examples("Sheng"); show_examples("Code-switching")

In [ ]:
show_examples("Ambiguity"); show_examples("Context-dependent"); show_examples("Mixed sentiment")

### Automated label-consistency screen (crude heuristic)
A small English/Kiswahili/Sheng keyword lexicon flags **obvious** conflicts, e.g. a *Positive* label on a message with only negative words and no sarcasm/negation flag. It will miss subtle cases and over-flag some. Treat it as a pointer for manual review, not ground truth.

In [ ]:
NEG = set("bad terrible awful worst angry disappointed useless slow failed failure error problem complaint refund unacceptable poor rude "
          "frustrated frustrating stuck delay delayed cheated scam mbaya vibaya mbovu tatizo shida hasira umenichosha imeharibika "
          "haifanyi sijapata sijafurahi ujinga kuchelewa wizi".split())
POS = set("good great excellent thanks thank love amazing perfect happy satisfied helpful fast quick wonderful appreciate "
          "asante nzuri safi poa vizuri nimefurahi mzuri bomba fiti shukrani sawa".split())
tok = lambda s: re.findall(r"[a-zA-Z']+", str(s).lower())
df["_neg_n"] = df[TEXT].map(lambda s: sum(w in NEG for w in tok(s)))
df["_pos_n"] = df[TEXT].map(lambda s: sum(w in POS for w in tok(s)))
sar = masks["Sarcasm"]; neg_flag = masks["Negation"]
susp_pos = df[(df["_sent"] == "Positive") & (df["_neg_n"] > 0) & (df["_pos_n"] == 0) & ~sar & ~neg_flag]
susp_neg = df[(df["_sent"] == "Negative") & (df["_pos_n"] > 0) & (df["_neg_n"] == 0) & ~sar & ~neg_flag]
print(f"Positive-labelled with only negative cue words: {len(susp_pos)}")
print(f"Negative-labelled with only positive cue words: {len(susp_neg)}")
display(susp_pos[[TEXT, SENT] + ([SCORE] if SCORE else [])].head(10))
display(susp_neg[[TEXT, SENT] + ([SCORE] if SCORE else [])].head(10))
for c in ["_neg_n", "_pos_n"]:
    pass
display(df.groupby("_sent")[["_pos_n", "_neg_n"]].mean().loc[ORDER].round(3).rename(columns={"_pos_n":"avg positive cue words","_neg_n":"avg negative cue words"}))

## 6. Operational analysis

In [ ]:
ops = [c for c in [DEPT, INQ, PRI, QUEUE, CHANNEL, STATUS] if c]
n = len(ops)
fig, axes = plt.subplots((n + 1) // 2, 2, figsize=(16, 4.2 * ((n + 1) // 2)))
for ax, c in zip(np.ravel(axes), ops):
    order = df[c].value_counts().index
    sns.countplot(y=df[c], order=order, ax=ax); ax.set_title(c)
for ax in np.ravel(axes)[n:]: ax.set_axis_off()
plt.tight_layout(); plt.show()

In [ ]:
for c in [DEPT, PRI, INQ]:
    if c:
        heat(pct_ct(c, "_sent")[ORDER], f"{c} × sentiment (row %)", figsize=(8, max(3, .5*df[c].nunique())))
        print(f"Cramér's V ({c} ↔ sentiment) = {cramers_v(c, '_sent'):.3f}")

if DEPT and INQ:
    ct = pd.crosstab(df[DEPT], df[INQ])
    heat(ct, "Department × inquiry type (counts)", fmt="d", figsize=(max(9, .9*ct.shape[1]), max(4, .5*ct.shape[0])))
    print("Top inquiry type within each department:")
    top = ct.idxmax(axis=1).to_frame("top_inquiry_type"); top["share_%"] = (ct.max(axis=1) / ct.sum(axis=1) * 100).round(1)
    display(top)
    print(f"Cramér's V (department ↔ inquiry type) = {cramers_v(DEPT, INQ):.3f}")

### Is negative sentiment simply = high priority?
If priority is just a relabelling of sentiment, a model could learn the shortcut and the "priority" feature adds no independent information. Sentiment (how the customer feels) and priority (how urgent the business problem is) should be correlated but **not** identical.

In [ ]:
if PRI:
    prio = df[PRI].astype(str).str.lower()
    HIGH = prio.str.contains("high|urgent|critical|p1|p0")
    LOW = prio.str.contains("low|p3|p4")
    print("Priority values:", df[PRI].unique().tolist())
    print("Treated as HIGH:", sorted(df.loc[HIGH, PRI].unique()), "| LOW:", sorted(df.loc[LOW, PRI].unique()))
    tbl = pd.DataFrame({"% high priority": HIGH.groupby(df["_sent"]).mean() * 100,
                        "% low priority": LOW.groupby(df["_sent"]).mean() * 100}).loc[ORDER].round(1)
    display(tbl)
    neg_high = (HIGH & (df["_sent"] == "Negative")).sum()
    print(f"P(high priority | Negative)        = {HIGH[df['_sent']=='Negative'].mean():.1%}")
    print(f"P(high priority | not Negative)    = {HIGH[df['_sent']!='Negative'].mean():.1%}")
    print(f"P(Negative | high priority)        = {(df['_sent'][HIGH]=='Negative').mean():.1%}")
    print(f"Cramér's V (priority ↔ sentiment)  = {cramers_v(PRI, '_sent'):.3f}")
    print("\nInconsistency screens (likely conflation of sentiment with urgency):")
    a = df[(df["_sent"] == "Negative") & LOW]
    b = df[(df["_sent"] == "Positive") & HIGH]
    c_ = df[(df["_sent"] == "Neutral") & HIGH]
    print(f" - Negative but LOW priority : {len(a)}  (legit: annoyed about a minor issue)")
    print(f" - Positive but HIGH priority: {len(b)}  (legit only if urgent request + happy tone)")
    print(f" - Neutral  but HIGH priority: {len(c_)}  (legit: factual but urgent, e.g. outage report)")
    cols = [c for c in [TEXT, SENT, PRI, DEPT, INQ] if c]
    for name, sub in [("Negative + LOW", a), ("Positive + HIGH", b), ("Neutral + HIGH", c_)]:
        if len(sub):
            print(f"\n{name} – sample:"); display(sub[cols].sample(min(5, len(sub)), random_state=3))
    if cramers_v(PRI, "_sent") > .5:
        print("\n⚠ Strong priority–sentiment coupling: sentiment and urgency look conflated in the generator.")
        issues.append("[operational] priority is strongly coupled to sentiment (Cramér's V>0.5). Proposal: do not use priority as a model feature; consider regenerating priority independently of sentiment.")

## 7. Numerical features

In [ ]:
num_cols = [c for c in [SCORE, INTENS, RESTIME, CONF] if c]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
desc = df[num_cols].agg(["min", "max", "mean", "median", "std", "skew"]).T
def iqr_out(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return int(((s < q1 - 1.5*i) | (s > q3 + 1.5*i)).sum())
desc["outliers_IQR"] = [iqr_out(df[c].dropna()) for c in num_cols]
desc["outliers_%"] = (desc["outliers_IQR"] / len(df) * 100).round(2)
display(desc.round(3))

fig, axes = plt.subplots(2, len(num_cols), figsize=(4.5 * len(num_cols), 7))
axes = np.atleast_2d(axes)
for j, c in enumerate(num_cols):
    sns.histplot(df[c].dropna(), kde=True, ax=axes[0, j]); axes[0, j].set_title(c)
    sns.boxplot(x=df[c], ax=axes[1, j])
plt.tight_layout(); plt.show()

if len(num_cols) > 1:
    heat(df[num_cols].corr(), "Correlation between numeric features", fmt=".2f", cmap="coolwarm", figsize=(6, 5))

### Does `sentiment_score` agree with the sentiment label?
Positive → mostly > 0, Negative → mostly < 0, Neutral → near 0. **Mixed is not forced to 0**: a Mixed message can contain strong positives and negatives, so any score is plausible; we only check that it is reported sensibly.

In [ ]:
if SCORE:
    lo, hi = df[SCORE].min(), df[SCORE].max()
    print(f"Score range: [{lo:.3f}, {hi:.3f}]")
    band = 0.2 * max(abs(lo), abs(hi))
    print(f"Neutral band used: |score| ≤ {band:.3f} (20% of the max absolute score; adjust if needed)")

    plt.figure(figsize=(10, 5))
    sns.violinplot(data=df, x="_sent", y=SCORE, order=ORDER, inner="quartile"); plt.axhline(0, color="k", ls="--", lw=.8)
    plt.title("sentiment_score by sentiment label"); plt.show()
    display(df.groupby("_sent")[SCORE].describe().loc[ORDER].round(3))

    s = df[SCORE]
    sc = df["_sent"]
    agree = pd.Series(np.select([sc == "Positive", sc == "Negative", sc == "Neutral"],
                                [(s > 0).astype(float), (s < 0).astype(float), (s.abs() <= band).astype(float)],
                                default=np.nan), index=df.index)
    rep = agree.groupby(df["_sent"]).agg(["mean", "count"]).rename(columns={"mean": "agreement_rate"})
    display(rep.loc[[c for c in ORDER if c != "Mixed"]].round(3))

    mixed = df[df["_sent"] == "Mixed"][SCORE]
    if len(mixed):
        print(f"Mixed: mean={mixed.mean():.3f}, std={mixed.std():.3f}, "
              f"{(mixed.abs() <= band).mean():.1%} near zero, {(mixed > band).mean():.1%} positive-leaning, {(mixed < -band).mean():.1%} negative-leaning")
        print("Interpretation: a wide spread is fine for Mixed. Concern only if Mixed scores are indistinguishable from Positive/Negative.")

    bad = df[agree == 0]
    print(f"\nDisagreeing rows (excluding Mixed): {len(bad)}")
    cols = [c for c in [TEXT, SENT, SCORE, EMO] if c]
    display(bad[cols].sample(min(10, len(bad)), random_state=5) if len(bad) else "none")
    if len(bad) / max(1, agree.notna().sum()) > .05:
        issues.append(f"[numeric] {len(bad)} rows where sentiment_score contradicts the label. Proposal: review; check whether sarcasm explains them before correcting.")

if CONF:
    print("\nAnnotation confidence by sentiment:")
    display(df.groupby("_sent")[CONF].describe().loc[ORDER].round(3))
    low_conf = df[df[CONF] < df[CONF].quantile(.05)]
    print(f"Bottom-5% confidence rows ({len(low_conf)}) – candidates for relabelling review. Sentiment mix:")
    print(low_conf["_sent"].value_counts(normalize=True).round(3).to_string())

## 8. Text analysis

In [ ]:
for c in [TEXT, CTX]:
    if c:
        t = df[c].fillna("").astype(str)
        df[f"_{c}_chars"] = t.str.len()
        df[f"_{c}_words"] = t.str.split().str.len()
len_cols = [c for c in df.columns if c.startswith("_") and c.endswith(("_chars", "_words"))]
display(df[len_cols].describe().T.round(2))

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(df[f"_{TEXT}_words"], bins=40, ax=ax[0]); ax[0].set_title("customer_inquiry – word count")
sns.kdeplot(data=df, x=f"_{TEXT}_words", hue="_sent", hue_order=ORDER, common_norm=False, ax=ax[1]); ax[1].set_title("word count by sentiment")
sns.boxplot(data=df, x="_sent", y=f"_{TEXT}_chars", order=ORDER, ax=ax[2]); ax[2].set_title("char count by sentiment")
plt.tight_layout(); plt.show()
if CTX:
    print(f"conversation_context empty/missing: {(df[CTX].fillna('').astype(str).str.strip()=='').mean():.1%}")

In [ ]:
SW_SW = set("na ya wa kwa ni la za katika kuwa hii hiyo yangu wangu pia au kama hapa mimi wewe yeye sisi nina una ana ili".split())
STOP = set(ENGLISH_STOP_WORDS) | SW_SW
words = lambda s: [w for w in re.findall(r"[a-zA-Z']+", str(s).lower()) if w not in STOP and len(w) > 1]

def top_words(series, k=20):
    c = Counter(w for s in series.dropna() for w in words(s)); return c.most_common(k)

for col in [c for c in [TEXT, CTX] if c]:
    tw = pd.DataFrame(top_words(df[col], 25), columns=["word", "count"])
    plt.figure(figsize=(9, 5)); sns.barplot(data=tw, y="word", x="count"); plt.title(f"Most frequent words – {col}"); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, len(ORDER), figsize=(4.5 * len(ORDER), 6))
for ax, s_ in zip(np.atleast_1d(axes), ORDER):
    tw = pd.DataFrame(top_words(df.loc[df["_sent"] == s_, TEXT], 12), columns=["word", "count"])
    sns.barplot(data=tw, y="word", x="count", ax=ax); ax.set_title(s_)
plt.suptitle("Top words by sentiment"); plt.tight_layout(); plt.show()

In [ ]:
def top_bigrams(series, k=15):
    cv = CountVectorizer(ngram_range=(2, 2), stop_words=list(STOP), min_df=2).fit(series.dropna().astype(str))
    X = cv.transform(series.dropna().astype(str)); s = np.asarray(X.sum(0)).ravel()
    idx = s.argsort()[::-1][:k]; return [(cv.get_feature_names_out()[i], int(s[i])) for i in idx]

bg = pd.DataFrame(top_bigrams(df[TEXT], 20), columns=["bigram", "count"])
plt.figure(figsize=(9, 6)); sns.barplot(data=bg, y="bigram", x="count"); plt.title("Most frequent bigrams – customer_inquiry"); plt.tight_layout(); plt.show()
for s_ in ORDER:
    print(f"\n{s_}:", top_bigrams(df.loc[df['_sent'] == s_, TEXT], 8))

# Template detection: how concentrated is the text?
top10 = df["_norm_text"].value_counts().head(10)
print("\nMost repeated inquiries (template check):"); print(top10.to_string())

## Save summary for later stages

In [ ]:
summary = {
    "rows": int(len(df)), "columns": int(df.shape[1]),
    "class_counts": {k: int(v) for k, v in vc.items()},
    "majority_minority_ratio": float(ratio),
    "unique_normalised_texts": int(df["_norm_text"].nunique()),
    "issues": issues,
}
os.makedirs("/kaggle/working", exist_ok=True)
with open("/kaggle/working/stage_2_eda_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2)[:2000])

## Notes
* The task brief supplied ended mid-way through **Section 8** ("Most frequent words / Common bigr…"). Any sections after that (e.g. a data-readiness summary or a model-training plan) still need to be added.
* Review the **issues log** in Section 2 and the manual-inspection examples in Section 5 before moving on to modelling. Label noise and templated text will cap your model quality more than the choice of architecture.